# Goodness of Fit Analysis

This notebook reads the generated result CSV files in `../results/Submitted_results` and recalculates goodness-of-fit statistics from the saved fitted curves. The resolution-adjusted and time-cut sensitivity-analysis files are excluded.

The result CSVs are exported with one row per time point. For the goodness-of-fit calculations below, each fitted curve is reconstructed by grouping rows by result file, `Type`, `comp`, and `Channel_number`.


## Metrics

For each fitted curve, the analysis compares `fitted_emission` with the observed `emission` values. The individual-fit metrics are normalised by the fitted peak height, `R_peak_fit`.

- `mean_r2_normalised_by_r_peak`: mean of the individual curve-level R-squared values.
- `mean_rmse_normalised_by_r_peak`: mean of the individual curve-level RMSE values divided by `abs(R_peak_fit)`.
- `r2_mean_fitted_vs_mean_emission`: R-squared comparing the mean fitted curve to the mean observed curve within the same `result_csv`, `Type`, and `comp`.
- `rmse_mean_fitted_vs_mean_emission`: RMSE comparing the mean fitted curve to the mean observed curve, normalised by the peak of that mean fitted curve.

R-squared is unchanged when both curves are divided by the same non-zero value. The mean-curve metrics are calculated within matched `Type` and `comp` groups before any broader averaging is done.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "results" / "Submitted_results").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

RESULTS_DIR = PROJECT_ROOT / "results" / "Submitted_results"
EXCLUDED_PATTERNS = ("resolution_adjusted", "time_cut_1percent")
CSV_FILES = [
    path
    for path in sorted(RESULTS_DIR.glob("Results_*.csv"))
    if not any(pattern in path.name for pattern in EXCLUDED_PATTERNS)
]

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", "{:.4f}".format)

print(f"Included result CSV files: {len(CSV_FILES)}")


Included result CSV files: 22


## Goodness-of-Fit Functions

In [2]:
def finite_pairs(observed, fitted):
    """Return finite observed/fitted pairs as NumPy arrays."""
    y = pd.to_numeric(observed, errors="coerce").to_numpy(dtype=float)
    y_hat = pd.to_numeric(fitted, errors="coerce").to_numpy(dtype=float)
    mask = np.isfinite(y) & np.isfinite(y_hat)
    return y[mask], y_hat[mask]


def r_squared(observed, fitted):
    """Coefficient of determination for one reconstructed fitted curve."""
    y, y_hat = finite_pairs(observed, fitted)
    if len(y) == 0:
        return np.nan

    ss_tot = np.sum((y - np.mean(y)) ** 2)
    if ss_tot <= 0:
        return np.nan

    ss_res = np.sum((y - y_hat) ** 2)
    return 1 - ss_res / ss_tot


def rmse(observed, fitted):
    """Root mean squared error for one reconstructed fitted curve."""
    y, y_hat = finite_pairs(observed, fitted)
    if len(y) == 0:
        return np.nan
    return np.sqrt(np.mean((y - y_hat) ** 2))


def first_valid_number(values):
    """Return the first finite number in a Series, or NaN if none exists."""
    numeric = pd.to_numeric(values, errors="coerce")
    numeric = numeric[np.isfinite(numeric)]
    if numeric.empty:
        return np.nan
    return float(numeric.iloc[0])

In [3]:
def analysis_class(result_csv):
    """Label ordinary and multiple-damage outputs internally."""
    if "trip" in result_csv:
        return "multiple_damage"
    return "ordinary"


def load_result_csv(path):
    """Load one submitted result CSV and coerce the curve columns to numeric values."""
    df = pd.read_csv(path, encoding="utf-8-sig")
    df["result_csv"] = path.name
    df["analysis_class"] = analysis_class(path.name)

    required = ["emission", "fitted_emission", "R_peak_fit", "Type", "comp", "Channel_number", "time"]
    missing = [col for col in required if col not in df.columns]
    if missing:
        raise ValueError(f"{path.name} is missing required columns: {missing}")

    for col in ["emission", "fitted_emission", "R_peak_fit", "time"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    return df


def calculate_curve_metrics(curve):
    """Calculate normalised goodness-of-fit metrics for one fitted curve."""
    curve = curve.sort_values("time", kind="mergesort")
    observed = curve["emission"]
    fitted = curve["fitted_emission"]
    r_peak = first_valid_number(curve["R_peak_fit"])

    basic_rmse = rmse(observed, fitted)

    if np.isfinite(r_peak) and r_peak != 0:
        norm_observed = observed / r_peak
        norm_fitted = fitted / r_peak
        norm_r2 = r_squared(norm_observed, norm_fitted)
        norm_rmse = basic_rmse / abs(r_peak)
    else:
        norm_r2 = np.nan
        norm_rmse = np.nan

    return pd.Series(
        {
            "R_peak_fit": r_peak,
            "r2_normalised_by_r_peak": norm_r2,
            "rmse_normalised_by_r_peak": norm_rmse,
        }
    )


def calculate_mean_curve_metrics(group):
    """Compare the mean fitted curve to the mean observed curve for one Type/comp group."""
    mean_curve = (
        group.groupby("time", dropna=True)
        .agg(
            mean_emission=("emission", "mean"),
            mean_fitted_emission=("fitted_emission", "mean"),
        )
        .reset_index()
        .sort_values("time", kind="mergesort")
    )

    mean_peak = np.nanmax(np.abs(mean_curve["mean_fitted_emission"]))
    mean_curve_r2 = r_squared(mean_curve["mean_emission"], mean_curve["mean_fitted_emission"])
    mean_curve_rmse = rmse(mean_curve["mean_emission"], mean_curve["mean_fitted_emission"])

    if np.isfinite(mean_peak) and mean_peak != 0:
        mean_curve_rmse_normalised = mean_curve_rmse / mean_peak
    else:
        mean_curve_rmse_normalised = np.nan

    return pd.Series(
        {
            "r2_mean_fitted_vs_mean_emission": mean_curve_r2,
            "rmse_mean_fitted_vs_mean_emission": mean_curve_rmse_normalised,
        }
    )


In [4]:
def calculate_all_fit_metrics(csv_files=CSV_FILES):
    """Return individual-fit and Type/comp mean-curve metrics for selected CSVs."""
    per_fit = []
    per_mean_curve = []
    skipped = []

    for path in csv_files:
        try:
            df = load_result_csv(path)

            fit_group_cols = ["result_csv", "analysis_class", "Type", "comp", "Channel_number"]
            fit_metrics = df.groupby(fit_group_cols, dropna=False, sort=False).apply(calculate_curve_metrics)
            per_fit.append(fit_metrics.reset_index())

            mean_group_cols = ["result_csv", "analysis_class", "Type", "comp"]
            mean_curve_metrics = df.groupby(mean_group_cols, dropna=False, sort=False).apply(calculate_mean_curve_metrics)
            per_mean_curve.append(mean_curve_metrics.reset_index())
        except Exception as exc:
            skipped.append({"result_csv": path.name, "error": str(exc)})

    if not per_fit:
        raise ValueError(f"No result CSV files could be processed in {RESULTS_DIR}")

    all_fit_metrics = pd.concat(per_fit, ignore_index=True)
    all_mean_curve_metrics = pd.concat(per_mean_curve, ignore_index=True)
    skipped = pd.DataFrame(skipped)
    return all_fit_metrics, all_mean_curve_metrics, skipped


def summarise_metrics(fit_metrics, mean_curve_metrics, group_cols):
    """Summarise individual fits and precomputed Type/comp mean-curve metrics."""
    fit_summary = (
        fit_metrics.groupby(group_cols, dropna=False)
        .agg(
            n_fits=("r2_normalised_by_r_peak", "size"),
            mean_r2_normalised_by_r_peak=("r2_normalised_by_r_peak", "mean"),
            mean_rmse_normalised_by_r_peak=("rmse_normalised_by_r_peak", "mean"),
        )
        .reset_index()
    )

    mean_summary = (
        mean_curve_metrics.groupby(group_cols, dropna=False)
        .agg(
            r2_mean_fitted_vs_mean_emission=("r2_mean_fitted_vs_mean_emission", "mean"),
            rmse_mean_fitted_vs_mean_emission=("rmse_mean_fitted_vs_mean_emission", "mean"),
        )
        .reset_index()
    )

    return fit_summary.merge(mean_summary, on=group_cols, how="left")


def summarise_total(fit_metrics, mean_curve_metrics, label="selected submitted result CSVs"):
    """Summarise all fit-level and Type/comp mean-curve metrics into one row."""
    return pd.DataFrame(
        [
            {
                "summary": label,
                "n_fits": len(fit_metrics),
                "mean_r2_normalised_by_r_peak": fit_metrics["r2_normalised_by_r_peak"].mean(),
                "mean_rmse_normalised_by_r_peak": fit_metrics["rmse_normalised_by_r_peak"].mean(),
                "r2_mean_fitted_vs_mean_emission": mean_curve_metrics["r2_mean_fitted_vs_mean_emission"].mean(),
                "rmse_mean_fitted_vs_mean_emission": mean_curve_metrics["rmse_mean_fitted_vs_mean_emission"].mean(),
            }
        ]
    )


## Submitted Result CSVs

Sensitivity-analysis CSVs are intentionally excluded here.

- [Results_CYP92C5_GeneExpression.csv](../results/Submitted_results/Results_CYP92C5_GeneExpression.csv)
- [Results_DMNT_Genotype.csv](../results/Submitted_results/Results_DMNT_Genotype.csv)
- [Results_DMNT_Single_OS.csv](../results/Submitted_results/Results_DMNT_Single_OS.csv)
- [Results_DMNT_dose.csv](../results/Submitted_results/Results_DMNT_dose.csv)
- [Results_DMNT_leaves.csv](../results/Submitted_results/Results_DMNT_leaves.csv)
- [Results_DMNT_single_herb.csv](../results/Submitted_results/Results_DMNT_single_herb.csv)
- [Results_DMNT_time.csv](../results/Submitted_results/Results_DMNT_time.csv)
- [Results_DMNT_trip.csv](../results/Submitted_results/Results_DMNT_trip.csv)
- [Results_HAC_HiRes.csv](../results/Submitted_results/Results_HAC_HiRes.csv)
- [Results_IGL_GeneExpression.csv](../results/Submitted_results/Results_IGL_GeneExpression.csv)
- [Results_TMTT_Single_OS.csv](../results/Submitted_results/Results_TMTT_Single_OS.csv)
- [Results_TMTT_single_herb.csv](../results/Submitted_results/Results_TMTT_single_herb.csv)
- [Results_TPS10_GeneExpression.csv](../results/Submitted_results/Results_TPS10_GeneExpression.csv)
- [Results_TPS2_GeneExpression.csv](../results/Submitted_results/Results_TPS2_GeneExpression.csv)
- [Results_hexa_HiRes.csv](../results/Submitted_results/Results_hexa_HiRes.csv)
- [Results_hexo_HiRes.csv](../results/Submitted_results/Results_hexo_HiRes.csv)
- [Results_indole_Single_OS.csv](../results/Submitted_results/Results_indole_Single_OS.csv)
- [Results_indole_single_herb.csv](../results/Submitted_results/Results_indole_single_herb.csv)
- [Results_mono_Single_OS.csv](../results/Submitted_results/Results_mono_Single_OS.csv)
- [Results_mono_single_herb.csv](../results/Submitted_results/Results_mono_single_herb.csv)
- [Results_sesq_Single_OS.csv](../results/Submitted_results/Results_sesq_Single_OS.csv)
- [Results_sesq_single_herb.csv](../results/Submitted_results/Results_sesq_single_herb.csv)


## Per-CSV Summary

The table below contains one row for each `result_csv` / `Type` / `comp` combination, followed by a total row for each CSV. Individual-fit values are averaged across fitted curves. Mean-curve values compare the mean fitted curve to the mean observed curve within matched `Type` and `comp` groups.


In [5]:
all_fit_metrics, all_mean_curve_metrics, skipped_files = calculate_all_fit_metrics()

by_file_type_comp = summarise_metrics(
    all_fit_metrics,
    all_mean_curve_metrics,
    ["result_csv", "analysis_class", "Type", "comp"],
)
by_file_total = summarise_metrics(
    all_fit_metrics,
    all_mean_curve_metrics,
    ["result_csv", "analysis_class"],
)
by_file_total["Type"] = "ALL"
by_file_total["comp"] = "ALL"

summary_columns = [
    "result_csv",
    "Type",
    "comp",
    "n_fits",
    "mean_r2_normalised_by_r_peak",
    "mean_rmse_normalised_by_r_peak",
    "r2_mean_fitted_vs_mean_emission",
    "rmse_mean_fitted_vs_mean_emission",
]

goodness_of_fit_summary = pd.concat(
    [by_file_type_comp[summary_columns], by_file_total[summary_columns]],
    ignore_index=True,
).sort_values(["result_csv", "Type", "comp"])

display(goodness_of_fit_summary)

if not skipped_files.empty:
    display(skipped_files)


,result_csv,Type,comp,n_fits,mean_r2_normalised_by_r_peak,mean_rmse_normalised_by_r_peak,r2_mean_fitted_vs_mean_emission,rmse_mean_fitted_vs_mean_emission
41,Results_CYP92C5_GeneExpression.csv,ALL,ALL,5,0.7710,0.1739,0.9616,0.0815
0,Results_CYP92C5_GeneExpression.csv,b,CYP92C5,5,0.7710,0.1739,0.9616,0.0815
42,Results_DMNT_Genotype.csv,ALL,ALL,11,0.9082,0.0743,0.9585,0.0488
1,Results_DMNT_Genotype.csv,CML287,DMNT,4,0.9888,0.0323,0.9976,0.0149
2,Results_DMNT_Genotype.csv,MO17,DMNT,4,0.7870,0.1279,0.8941,0.0910
3,Results_DMNT_Genotype.csv,NC3000,DMNT,3,0.9623,0.0588,0.9838,0.0404
43,Results_DMNT_Single_OS.csv,ALL,ALL,10,0.9089,0.0885,0.9752,0.0451
4,Results_DMNT_Single_OS.csv,w,DMNT,5,0.9717,0.0557,0.9949,0.0241
5,Results_DMNT_Single_OS.csv,wo,DMNT,5,0.8462,0.1212,0.9555,0.0660
44,Results_DMNT_dose.csv,ALL,ALL,30,0.8764,0.0891,0.9595,0.0509


## Total Summary

The final summaries include the selected CSVs in `../results/Submitted_results`, excluding the resolution-adjusted and time-cut sensitivity-analysis files.


In [6]:
total_average_goodness_of_fit = summarise_total(all_fit_metrics, all_mean_curve_metrics)
goodness_of_fit_by_comp = summarise_metrics(all_fit_metrics, all_mean_curve_metrics, ["comp"]).sort_values("comp")

display(total_average_goodness_of_fit)
display(goodness_of_fit_by_comp)


,summary,n_fits,mean_r2_normalised_by_r_peak,mean_rmse_normalised_by_r_peak,r2_mean_fitted_vs_mean_emission,rmse_mean_fitted_vs_mean_emission
0,selected submitted result CSVs,212,0.8154,0.1285,0.9419,0.0663


,comp,n_fits,mean_r2_normalised_by_r_peak,mean_rmse_normalised_by_r_peak,r2_mean_fitted_vs_mean_emission,rmse_mean_fitted_vs_mean_emission
0,CYP92C5,5,0.7710,0.1739,0.9616,0.0815
1,DMNT,111,0.9088,0.0829,0.9711,0.0442
2,HAC,3,0.9804,0.0464,0.9865,0.0384
3,IGL,5,0.9290,0.0761,0.9686,0.0589
4,TMTT,19,0.5629,0.2580,0.8698,0.1112
5,TPS10,4,0.7500,0.1856,0.8668,0.1321
6,TPS2,5,0.6074,0.2434,0.8076,0.1440
7,hexa,3,0.9974,0.0120,0.9983,0.0108
8,hexo,3,0.9894,0.0317,0.9908,0.0296
9,indole,19,0.7750,0.1284,0.9358,0.0672


## Results and Discussion

Across the selected submitted result files, individual fitted curves showed good overall agreement with the observed data, with a mean peak-normalised R-squared of 0.82 and a mean peak-normalised RMSE of 0.13. When fit quality was assessed at the group-mean level, by comparing the mean fitted response curve with the mean observed emission curve within each matching `Type` and `comp` group, performance was substantially stronger: the mean-curve R-squared increased to 0.94 and the mean-curve normalised RMSE decreased to 0.07. This suggests that much of the lower goodness of fit at the individual-replicate level reflects replicate-level noise and biological variability, rather than a general failure of the model to capture the central response dynamics. This distinction is particularly important for noisier compounds, where individual replicate R-squared values may underestimate how well the model describes the average biological response.
